# MuZero · Planning Lab
SPDX-License-Identifier: Apache-2.0

Run a finite CPU learning experiment and inspect measured results. No API key or public tunnel.

[Project notice](https://github.com/MontrealAI/AGI-Alpha-Agent-v0/blob/main/docs/DISCLAIMER_SNIPPET.md).
The original notebook is preserved in `archive/colab.original.ipynb`.

In [ ]:
from pathlib import Path
import os, subprocess, sys
root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "alpha_factory_v1/demos/muzero_planning/training.py").is_file()), None)
if root is None:
    root = Path.cwd() / "AGI-Alpha-Agent-v0"
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/MontrealAI/AGI-Alpha-Agent-v0.git", str(root)], check=True)
os.chdir(root)
sys.path.insert(0, str(root))
print("Source commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())

Install the dedicated dependency lock. This cell needs network access unless you have a wheelhouse. It can take several minutes; it does not download model weights.

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "--require-hashes", "-r", "alpha_factory_v1/demos/muzero_planning/requirements.lock"], check=True)

In [ ]:
from alpha_factory_v1.demos.muzero_planning.training import Config, experiment
config = Config(env_id="MiniChoice-v0", seed=42, episodes=32, simulations=32, max_steps=8)
for report in experiment(config):
    if report["status"] == "training":
        print(f"Episode {report['completed']}/{report['total']}")
print(report["evaluation"])
print(report["search_after"])

Compare random, untrained search, trained policy and trained search on the same held-out seeds. Returns on this small task do not imply CartPole mastery or AGI. Change the configuration deliberately; larger requests are bounded.

In [ ]:
import json
output = Path("muzero-experiment.json")
with output.open("x", encoding="utf-8") as handle:
    json.dump(report, handle, indent=2, allow_nan=False)
print(output.resolve())
# Existing files are never overwritten; choose a new name to export another run.